# L42 · 🎨 FFT 图形化 — 蝴蝶图 + 纯音 / 和弦 / 噪声的频谱形态对比

**目标**：用图直觉建立 DFT 矩阵、蝶形网络、频谱对称性的视觉记忆，
理解快速傅里叶变换「分而治之」背后的几何结构。

🔗 Aurora 连接：`aurora.audio.transforms.fft` · `aurora.audio.io.sine`

## 核心直觉

DFT 本质是一次矩阵-向量乘法：把时域信号投影到 N 个不同频率的复指数基上。FFT 的魔法在于这个 N×N 矩阵有高度冗余——把它递归地折叠成两半，计算量从 `O(N²)` 降到 `O(N log N)`，每一层折叠对应图中的一组蝶形操作。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine
from aurora.audio.transforms import fft

## 1. DFT 矩阵：N×N 酉矩阵

DFT 定义：`X[k] = sum_{n=0}^{N-1} x[n] * exp(-2*pi*i*k*n/N)`

写成矩阵形式，令 `W = exp(-2*pi*i/N)`，则

```
F[k, n] = W^(k*n),   k, n = 0, 1, ..., N-1
```

`F` 是 N×N **酉矩阵**（`F @ conj(F).T = N * I`），每一行是一个频率 k 的复指数，实部和虚部分别是余弦/正弦波。
热力图的行是频率 bin，列是时间采样点；颜色编码实部（余弦）。

In [ ]:
N = 16
n = np.arange(N)
k = np.arange(N)
# DFT 矩阵，形状 (N, N)
W = np.exp(-2j * np.pi * np.outer(k, n) / N)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(W.real, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[0].set_title(f'DFT 矩阵实部（余弦）  N={N}')
axes[0].set_xlabel('时间采样点 n')
axes[0].set_ylabel('频率 bin k')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(W.imag, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[1].set_title(f'DFT 矩阵虚部（正弦）  N={N}')
axes[1].set_xlabel('时间采样点 n')
axes[1].set_ylabel('频率 bin k')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# 验证酉性：F @ conj(F).T = N * I
err = np.max(np.abs(W @ W.conj().T - N * np.eye(N)))
print(f'酉性误差 max|F·F†  - N·I| = {err:.2e}  ✅' if err < 1e-10
      else f'酉性误差 {err:.2e}  ❌')

## 2. 蝶形网络：分而治之

Cooley-Tukey FFT 把长度 N 的 DFT 拆成两个 N/2 的 DFT：

```
X[k]     = E[k] + W^k * O[k]
X[k+N/2] = E[k] - W^k * O[k]
```

其中 `E[k]` 是偶数索引输入的 N/2 点 DFT，`O[k]` 是奇数索引的，
`W^k = exp(-2*pi*i*k/N)` 称为**旋转因子（twiddle factor）**。

N=8 时共有 `log2(8) = 3` 层，每层 4 个蝶形单元。
下图用节点-连线方式画出数据流，颜色标识旋转因子的相位。

In [ ]:
# 画 N=8 蝶形网络示意图（3 层，每层 4 个蝶形）
N = 8
stages = int(np.log2(N))  # 3

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-0.3, stages + 0.3)
ax.set_ylim(-0.5, N - 0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('N=8 蝶形网络（3 层 × 4 蝶形）', fontsize=13)

# bit-reversal 置换顺序（输入顺序）
bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(N)]

# 节点 y 坐标（输入侧按 bit-reversal 排列）
y_pos = {i: bit_rev[i] for i in range(N)}
# 从第0层到第3层均匀铺开
colors = plt.cm.plasma(np.linspace(0.15, 0.85, stages))

def node_xy(stage, wire):
    return stage, wire

# 画输入标签（bit-reversal 后的顺序）
for i in range(N):
    ax.text(-0.15, bit_rev[i], f'x[{i}]', va='center', ha='right', fontsize=8)

# 逐层画蝶形
current_y = list(range(N))  # 追踪每条 wire 的 y 坐标
for s in range(stages):
    half = 2 ** s          # 蝶形跨度
    group = 2 * half       # 组大小
    n_butterflies = N // group  # 每层蝶形组数
    col_in  = s
    col_out = s + 1
    for g in range(n_butterflies):
        for b in range(half):
            top = g * group + b
            bot = top + half
            yt, yb = top, bot
            # 旋转因子相位
            phase = b / group  # 0..0.5
            c = colors[s]
            # 直通线（上）
            ax.annotate('', xy=(col_out, yt), xytext=(col_in, yt),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # 直通线（下）
            ax.annotate('', xy=(col_out, yb), xytext=(col_in, yb),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # 交叉线 top→bot
            ax.plot([col_in, col_out], [yt, yb], color=c, lw=0.8, ls='--', alpha=0.6)
            # 旋转因子标注
            mid_x = (col_in + col_out) / 2
            mid_y = (yt + yb) / 2
            if b == 0:
                ax.text(mid_x, mid_y + 0.15,
                        f'W⁸^{b}', fontsize=6.5, color=c,
                        ha='center', va='bottom')

# 输出标签
for i in range(N):
    ax.text(stages + 0.15, i, f'X[{i}]', va='center', ha='left', fontsize=8)

# 层标签
for s in range(stages):
    ax.text(s + 0.5, N - 0.1, f'第{s+1}层', ha='center', fontsize=9,
            color=colors[s], fontweight='bold')

plt.tight_layout()
plt.show()
print(f'N={N}，共 {stages} 层，每层 {N//2} 个蝶形，总操作数 = {N//2 * stages}（vs 朴素 DFT {N*N}）')

## 3. 频谱对称：实信号的镜像结构

当输入 `x[n]` 是**实数**时，DFT 满足共轭对称：

```
X[N-k] = conj(X[k]),   k = 1, 2, ..., N/2-1
```

因此幅度谱 `|X[k]|` 关于 `k = N/2` 对称，只有前半段（`k = 0..N/2`）包含独立信息。
可视化中你会看到：左半段和右半段幅度是镜像；相位则反号。
Aurora 的 `fft` 输出全 N 个 bin，使用时取 `[:N//2+1]` 即可。

In [ ]:
# 用 aurora.audio.transforms.fft 演示共轭对称
sr = 8000
duration = 0.016  # 16 ms → N=128 采样（2^7，aurora FFT 需要 2 的幂）
x = sine(440.0, sample_rate=sr, duration=duration)  # 实数信号
N = len(x)

X = fft(x)          # aurora FFT，返回复数谱
mag   = np.abs(X)
phase = np.angle(X)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
bins = np.arange(N)

axes[0].stem(bins, mag, markerfmt='C0o', linefmt='C0-', basefmt='k-')
axes[0].axvline(N//2, color='red', ls='--', lw=1, label=f'k=N/2={N//2}')
axes[0].set_ylabel('幅度 |X[k]|')
axes[0].set_title(f'440 Hz 实信号的幅度谱（N={N}）——关于 k=N/2 对称')
axes[0].legend()

axes[1].stem(bins, phase, markerfmt='C1o', linefmt='C1-', basefmt='k-')
axes[1].axvline(N//2, color='red', ls='--', lw=1)
axes[1].set_ylabel('相位 ∠X[k]  (rad)')
axes[1].set_xlabel('频率 bin k')
axes[1].set_title('相位谱——右半段是左半段的镜像反号')

plt.tight_layout()
plt.show()

# 数值验证：X[N-k] = conj(X[k])
err = np.max(np.abs(X[1:N//2] - np.conj(X[N-1:N//2:-1])))
print(f'共轭对称误差 max|X[N-k] - conj(X[k])| = {err:.2e}',
      '✅' if err < 1e-10 else '❌')

## 参数实验：双频叠加信号

**信号参数**：`sr=8000`，`duration=0.05`（50 ms，N=400），
叠加 `f1=440 Hz`（bin ≈ 2.75）和 `f2=880 Hz`（bin ≈ 5.5）。

**预期现象**：
- 幅度谱出现两个对称峰，峰位 bin 约为 `round(f * N / sr)`
- 880 Hz 的 bin 位置恰好是 440 Hz 的两倍（八度关系）
- 超出 `N/2` 的镜像峰对称出现在高频侧

In [ ]:
sr       = 8000
duration = 0.032   # 32 ms → N=256（2^8）
f1, f2   = 440.0, 880.0

x1 = sine(f1, sample_rate=sr, duration=duration)
x2 = sine(f2, sample_rate=sr, duration=duration)
x  = x1 + x2
N  = len(x)

X   = fft(x)
mag = np.abs(X)

# 理论峰值 bin
bin1 = round(f1 * N / sr)
bin2 = round(f2 * N / sr)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mag, color='steelblue', lw=1.2)
ax.axvline(bin1, color='C1', ls='--', label=f'{f1:.0f} Hz  bin={bin1}')
ax.axvline(N - bin1, color='C1', ls=':', alpha=0.5, label=f'镜像 bin={N-bin1}')
ax.axvline(bin2, color='C2', ls='--', label=f'{f2:.0f} Hz  bin={bin2}')
ax.axvline(N - bin2, color='C2', ls=':', alpha=0.5, label=f'镜像 bin={N-bin2}')
ax.axvline(N//2, color='red', ls='-', lw=0.8, alpha=0.4, label=f'N/2={N//2}')
ax.set_xlabel('频率 bin k')
ax.set_ylabel('幅度 |X[k]|')
ax.set_title(f'440 Hz + 880 Hz 叠加信号频谱  (sr={sr}, N={N})')
ax.legend()
plt.tight_layout()
plt.show()

print(f'440 Hz → bin {bin1}  （理论：{f1*N/sr:.2f}）')
print(f'880 Hz → bin {bin2}  （理论：{f2*N/sr:.2f}）')
print(f'两峰 bin 比值：{bin2}/{bin1} = {bin2/bin1:.2f}  （应≈2.0，八度）')

## 本课小结

`fft`（来自 `aurora.audio.transforms`）把 N 点实信号映射为 N 个复数 bin，
通过 DFT 矩阵可视化可以看到每一行是一个复指数基，蝶形网络则揭示了
`O(N log N)` 的递归折叠结构，共轭对称保证实信号只需保留前 `N//2+1` 个 bin。
下节 `day1_stft.ipynb` 将用滑动窗口把 FFT 扩展到时频谱（STFT），
看频率随时间演变的全貌。